In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

dataset = pd.read_csv('./data/processed.csv')

X = dataset.drop('Attrition', axis=1)
y = dataset['Attrition']
X.head()

,Age,Gender,Years at Company,Monthly Income,Work-Life Balance,Job Satisfaction,Performance Rating,Number of Promotions,Distance from Home,Education Level,...,Job Role_Technology,Overtime_No,Overtime_Yes,Leadership Opportunities_No,Leadership Opportunities_Yes,Marital Status_Divorced,Marital Status_Married,Marital Status_Single,Remote Work_No,Remote Work_Yes
0,-0.623149,0,0.292097,-0.887054,1.493723,-0.921689,0.066924,1.172597,-0.981699,-0.572159,...,0,1,0,1,0,0,1,0,1,0
1,1.694084,1,-1.044365,-0.820155,-1.705401,0.223742,-2.636070,2.177337,-1.016770,1.162526,...,0,1,0,1,0,1,0,0,1,0
2,-1.202458,1,-0.509780,0.399360,0.427349,0.223742,-2.636070,-0.836883,-1.367482,0.295184,...,0,1,0,1,0,0,1,0,1,0
3,-0.209358,1,-0.777072,-1.537927,0.427349,0.223742,1.418420,0.167857,-0.806343,-1.439501,...,0,1,0,1,0,0,0,1,0,1
4,1.445809,0,2.252240,-1.151399,-0.639026,1.369173,0.066924,-0.836883,0.736790,-1.439501,...,0,0,1,1,0,1,0,0,1,0


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score

model = []
fold = [4, 5, 6]

def calc_metrics(X_train, X_valid, y_train, y_valid, model):
  train_pred = model.predict_proba(X_train)[:,1]
  valid_pred = model.predict_proba(X_valid)[:,1]

  mse_train = mean_squared_error(y_train, train_pred)
  r2_train = r2_score(y_train, train_pred)
  mse_valid = mean_squared_error(y_valid, valid_pred)
  r2_valid = r2_score(y_valid, valid_pred)

  arr = np.array([[mse_train, r2_train], [mse_valid, r2_valid]])
  return arr

def kfold(X, Y, model, splits, dim_reduction=None):
  kf = KFold(splits, shuffle=True, random_state=42)
  metrics = np.zeros((2, 2))
  for train_indices, valid_indices in kf.split(X, Y):

    X_train, X_valid, Y_train, Y_valid = [X.iloc[train_indices], X.iloc[valid_indices], Y[train_indices], Y[valid_indices]]
    if dim_reduction != None:
      X_train = dim_reduction.fit_transform(X_train)
      X_valid = dim_reduction.transform(X_valid)
    model.fit(X_train, Y_train)
    metrics += calc_metrics(X_train, X_valid, Y_train, Y_valid, model)

  metrics /= splits   
  print(model) 
  display(pd.DataFrame([['train', metrics[0,0], metrics[0,1]], ['validation', metrics[1,0], metrics[1,1]]], columns=['set', 'MSE', 'R2']))
  return model
  
hidden_layer = (32,)

In [3]:
for f in fold:
  print(f'------------------------------------{f} folds------------------------------------')
  model.append(kfold(X, y, LogisticRegression(max_iter=1000), f))
  model.append(kfold(X, y, MLPClassifier(hidden_layer_sizes=hidden_layer, max_iter=1000, learning_rate='adaptive'), f))

------------------------------------4 folds------------------------------------
LogisticRegression(max_iter=1000)


,set,MSE,R2
0,train,0.168779,0.323162
1,validation,0.168998,0.322268


MLPClassifier(hidden_layer_sizes=(32,), learning_rate='adaptive', max_iter=1000)


,set,MSE,R2
0,train,0.159733,0.359437
1,validation,0.164682,0.339577


------------------------------------5 folds------------------------------------
LogisticRegression(max_iter=1000)


...

,set,MSE,R2
0,train,0.160286,0.357219
1,validation,0.164213,0.341459


------------------------------------6 folds------------------------------------
LogisticRegression(max_iter=1000)


,set,MSE,R2
0,train,0.168791,0.323114
1,validation,0.168993,0.322269


MLPClassifier(hidden_layer_sizes=(32,), learning_rate='adaptive', max_iter=1000)


,set,MSE,R2
0,train,0.159333,0.361039
1,validation,0.163955,0.342474


In [ ]:
from sklearn.decomposition import PCA
for f in fold:
  print(f'------------------------------------{f} folds------------------------------------')
  model.append(kfold(pd.DataFrame(X), y, LogisticRegression(max_iter=1000), f, PCA(n_components=X.shape[1] // 3)))
  model.append(kfold(pd.DataFrame(X), y, MLPClassifier(hidden_layer_sizes=hidden_layer, max_iter=1000, learning_rate='adaptive'), f, PCA(n_components=X.shape[1] // 3)))
  # worse perfomance seems to do with few features

------------------------------------4 folds------------------------------------
LogisticRegression(max_iter=1000)


,set,MSE,R2
0,train,0.215072,0.137515
1,validation,0.214545,0.139613


MLPClassifier(hidden_layer_sizes=(32,), learning_rate='adaptive', max_iter=1000)


,set,MSE,R2
0,train,0.211790,0.150677
1,validation,0.212659,0.147177


------------------------------------5 folds------------------------------------
LogisticRegression(max_iter=1000)


...

,set,MSE,R2
0,train,0.212055,0.149615
1,validation,0.213446,0.144023


------------------------------------6 folds------------------------------------
LogisticRegression(max_iter=1000)


,set,MSE,R2
0,train,0.211460,0.152000
1,validation,0.211425,0.152101


MLPClassifier(hidden_layer_sizes=(32,), learning_rate='adaptive', max_iter=1000)


,set,MSE,R2
0,train,0.209231,0.160941
1,validation,0.210384,0.156278
